# Construction Domain Benchmark
## Three-Way Model Comparison: Base vs Generator-Tuned vs Judge-Tuned

> **IBM TechXchange Hackathon 2026 — Barrett-Aligned LoRA Fine-Tuning Demonstration**

---

## Section 1 — Theoretical Foundation

### Barrett's Concept-as-Population Model

Lisa Feldman Barrett's constructionist theory holds that a concept is not a fixed definition but a **population of variable instances**, each anchored to a specific context and a specific goal. The brain selects the functionally adequate instance — the one that best predicts the upcoming experience — rather than retrieving a stored prototype.

Current LLMs store token co-occurrence statistics. They have no mechanism to select the contextually adequate instance of a concept for a given goal. This project introduces the missing layer:

1. **Self-generate** a labelled corpus of `(context, goal, simulation, adequacy_score)` tuples using the Barrett-structured RL workflow.
2. **Fine-tune** two models from the same corpus:
   - **Generator-tuned**: trained on `(context, goal) → simulation` pairs (high-quality instances only, score ≥ 8.0)
   - **Judge-tuned**: trained on `(context, goal, simulation) → adequacy_score` pairs (full score range)
3. **Evaluate**: compare all three models — base, generator-tuned, judge-tuned — on held-out construction-domain concept terms.

### Construction Domain Scope

The domain is the **construction industry** — 10 concept terms spanning physical objects (scaffolding, formwork), processes (curing, tendering), relational concepts (tolerances, liability), and safety concepts (PPE, site induction).

This domain is narrow enough that 50 training examples produce a visible domain shift, and specific enough that the base model's generic priors form a meaningful comparison baseline.

### What the Deltas Demonstrate

- **Generator delta** (base vs generator-tuned generation quality): did training on Barrett-structured `(context, goal) → simulation` pairs make the model produce better construction-domain simulations?
- **Judge delta** (base vs judge-tuned scoring accuracy): did training on `(context, goal, simulation) → score` pairs make the model score construction-domain functional adequacy more accurately?

Two independent deltas from the same corpus make Barrett's claim empirically testable from two angles.

---
## Section 2 — Setup

In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
load_dotenv()

# ── Add repo root to path so src.* imports work from notebook ─────────────
_REPO_ROOT = Path().resolve().parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

_DATA_DIR = _REPO_ROOT / 'data'
_EVAL_FILE = _DATA_DIR / 'construction_eval.jsonl'
_CONFIG_FILE = _DATA_DIR / 'tuning_job_config.json'

print(f'Repo root : {_REPO_ROOT}')
print(f'Eval file : {_EVAL_FILE} (exists={_EVAL_FILE.exists()})')
print(f'Config    : {_CONFIG_FILE} (exists={_CONFIG_FILE.exists()})')

In [ ]:
# ── Load model IDs ────────────────────────────────────────────────────────
BASE_MODEL_ID = os.getenv('WATSONX_MODEL_ID', 'ibm/granite-3b-code-instruct')

# Try tuning_job_config.json first, fall back to env vars
if _CONFIG_FILE.exists():
    with open(_CONFIG_FILE) as f:
        _cfg = json.load(f)
    JUDGE_MODEL_ID = (
        _cfg.get('judge_job', {}).get('tuned_model_id')
        or os.getenv('WATSONX_JUDGE_MODEL_ID', BASE_MODEL_ID)
    )
    GENERATOR_MODEL_ID = (
        _cfg.get('generator_job', {}).get('tuned_model_id')
        or os.getenv('WATSONX_GENERATOR_MODEL_ID', BASE_MODEL_ID)
    )
else:
    JUDGE_MODEL_ID = os.getenv('WATSONX_JUDGE_MODEL_ID', BASE_MODEL_ID)
    GENERATOR_MODEL_ID = os.getenv('WATSONX_GENERATOR_MODEL_ID', BASE_MODEL_ID)

print(f'Base model      : {BASE_MODEL_ID}')
print(f'Judge model     : {JUDGE_MODEL_ID}')
print(f'Generator model : {GENERATOR_MODEL_ID}')

if JUDGE_MODEL_ID == BASE_MODEL_ID:
    print('\n⚠️  JUDGE_MODEL_ID == BASE_MODEL_ID — training may not have completed yet.')
    print('   Set WATSONX_JUDGE_MODEL_ID in .env after training finishes.')
if GENERATOR_MODEL_ID == BASE_MODEL_ID:
    print('\n⚠️  GENERATOR_MODEL_ID == BASE_MODEL_ID — training may not have completed yet.')
    print('   Set WATSONX_GENERATOR_MODEL_ID in .env after training finishes.')

In [ ]:
# ── Load eval set ─────────────────────────────────────────────────────────
def load_eval_instances(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(
            f'Eval file not found: {path}\n'
            'Run src/export_training_data.py first.'
        )
    instances = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                instances.append(json.loads(line))
    return instances

EVAL_INSTANCES = load_eval_instances(_EVAL_FILE)
print(f'Loaded {len(EVAL_INSTANCES)} eval instances:')
for inst in EVAL_INSTANCES:
    print(f"  • {inst['term']:25s}  score={inst.get('adequacy_score', 'N/A')}")

---
## Section 3 — Generation Comparison

We select 3 eval terms and run **one generation** with each of the three models (base, generator-tuned, judge-tuned). This directly answers: *does the generator-tuned model produce more goal-anchored construction-domain simulations?*

We expect:
- **Base model**: generic output, may not mention construction-specific processes or actors
- **Generator-tuned model**: explicitly references construction domain terms, goal-indexed actions
- **Judge-tuned model**: was not trained to generate — serves as a controlled comparison showing that domain fine-tuning *on the wrong objective* does not improve generation

In [ ]:
import importlib
import src.concept_loop as concept_loop

# Select 3 terms for generation comparison
GENERATION_TERMS = ['scaffolding', 'practical completion', 'liability']
generation_eval = [i for i in EVAL_INSTANCES if i['term'] in GENERATION_TERMS]

def generate_simulation(model_id: str, inst: Dict[str, Any]) -> str:
    """Generate one simulation for the given eval instance using the specified model."""
    population = concept_loop.run_rl_loop(
        term=inst['term'],
        context_goal_pairs=[(inst['context'], inst['goal'])],
        seed_phrase=inst.get('seed_phrase', ''),
        grammatical_frame=inst.get('grammatical_frame', ''),
        max_iterations=1,
        threshold=10.0,  # force exactly 1 round — no refinement
        model_id=model_id,
    )
    if population.instances:
        return population.instances[0].simulation
    return '[no simulation generated]'

print(f'Running generation comparison for {len(generation_eval)} terms × 3 models…')
print('(This makes real API calls unless WATSONX_STUB=true is set)\n')

In [ ]:
generation_results = []  # list of dicts with term, context, goal, base/generator/judge simulations

for inst in generation_eval:
    term = inst['term']
    print(f'Generating for: {term}')

    base_sim = generate_simulation(BASE_MODEL_ID, inst)
    print(f'  base       ✓')

    gen_sim = generate_simulation(GENERATOR_MODEL_ID, inst)
    print(f'  generator  ✓')

    judge_sim = generate_simulation(JUDGE_MODEL_ID, inst)
    print(f'  judge      ✓')

    generation_results.append({
        'term': term,
        'context': inst['context'],
        'goal': inst['goal'],
        'base_simulation': base_sim,
        'generator_simulation': gen_sim,
        'judge_simulation': judge_sim,
    })

print('\nGeneration complete.')

In [ ]:
# ── Display side-by-side simulation table ────────────────────────────────
import pandas as pd
from IPython.display import display, HTML

gen_df = pd.DataFrame([
    {
        'Term': r['term'],
        'Context (truncated)': r['context'][:80] + '…',
        'Goal': r['goal'][:60] + '…',
        'Base Model': r['base_simulation'],
        'Generator-Tuned': r['generator_simulation'],
        'Judge-Tuned': r['judge_simulation'],
    }
    for r in generation_results
])

print('### Generation Comparison Table')
display(gen_df.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))

---
## Section 4 — Adequacy Scoring Comparison

We score all 10 eval instances using **four scoring combinations**:

| Combination | Generator | Judge |
|---|---|---|
| A | Base | Base |
| B | Base | Judge-tuned |
| C | Generator-tuned | Base |
| D | Generator-tuned | Judge-tuned |

This cross-product separates the **generation quality signal** (A vs C) from the **judging quality signal** (A vs B), and shows how they interact (D).

We expect:
- **C > A**: generator-tuned model produces simulations that score higher even with the base judge
- **B ≠ A**: judge-tuned model scores differently from base judge (ideally more accurate for domain)
- **D highest**: best-of-both — domain-tuned generator + domain-tuned judge

In [ ]:
import src.judge as judge_module

def score_simulation(
    judge_model_id: str,
    inst: Dict[str, Any],
    simulation: str,
) -> Optional[float]:
    """Score a simulation using the specified judge model."""
    return judge_module.score_instance(
        term=inst['term'],
        context=inst['context'],
        goal=inst['goal'],
        simulation=simulation,
        seed_phrase=inst.get('seed_phrase', ''),
        grammatical_frame=inst.get('grammatical_frame', ''),
        model_id=judge_model_id,
    )

print('Running adequacy scoring for all eval instances × 4 combinations…')
print('(This makes real API calls unless WATSONX_STUB=true is set)\n')

In [ ]:
scoring_results = []

for inst in EVAL_INSTANCES:
    term = inst['term']
    print(f'Scoring: {term}')

    # Generate simulations with base and generator-tuned models
    base_sim = generate_simulation(BASE_MODEL_ID, inst)
    gen_sim = generate_simulation(GENERATOR_MODEL_ID, inst)

    # Score all four combinations
    score_A = score_simulation(BASE_MODEL_ID, inst, base_sim)        # base gen, base judge
    score_B = score_simulation(JUDGE_MODEL_ID, inst, base_sim)       # base gen, tuned judge
    score_C = score_simulation(BASE_MODEL_ID, inst, gen_sim)         # tuned gen, base judge
    score_D = score_simulation(JUDGE_MODEL_ID, inst, gen_sim)        # tuned gen, tuned judge

    scoring_results.append({
        'term': term,
        'base_sim': base_sim,
        'gen_sim': gen_sim,
        'A_base_gen_base_judge': score_A,
        'B_base_gen_tuned_judge': score_B,
        'C_tuned_gen_base_judge': score_C,
        'D_tuned_gen_tuned_judge': score_D,
        'training_score': inst.get('adequacy_score'),  # original training label
    })
    print(f'  A={score_A}  B={score_B}  C={score_C}  D={score_D}')

print('\nScoring complete.')

---
## Section 5 — Summary Delta Table

One row per term. Columns show:
- **gen_delta**: how much better the generator-tuned model scores vs base (Combination C − A)
- **judge_delta**: how much the judge-tuned model's scores differ from base judge (B − A)
- **best_combo**: highest score across all four combinations

In [ ]:
def _fmt(v):
    return f'{v:.2f}' if v is not None else 'N/A'

def _delta(a, b):
    if a is not None and b is not None:
        return round(b - a, 2)
    return None

summary_rows = []
for r in scoring_results:
    gen_delta = _delta(r['A_base_gen_base_judge'], r['C_tuned_gen_base_judge'])
    judge_delta = _delta(r['A_base_gen_base_judge'], r['B_base_gen_tuned_judge'])
    scores = [r['A_base_gen_base_judge'], r['B_base_gen_tuned_judge'],
              r['C_tuned_gen_base_judge'], r['D_tuned_gen_tuned_judge']]
    valid_scores = [s for s in scores if s is not None]
    best = max(valid_scores) if valid_scores else None

    summary_rows.append({
        'Term': r['term'],
        'A: base/base': _fmt(r['A_base_gen_base_judge']),
        'B: base/tuned-judge': _fmt(r['B_base_gen_tuned_judge']),
        'C: tuned-gen/base': _fmt(r['C_tuned_gen_base_judge']),
        'D: tuned-gen/tuned-judge': _fmt(r['D_tuned_gen_tuned_judge']),
        'Gen Δ (C−A)': _fmt(gen_delta),
        'Judge Δ (B−A)': _fmt(judge_delta),
        'Best combo': _fmt(best),
        'Training score': _fmt(r.get('training_score')),
    })

summary_df = pd.DataFrame(summary_rows)

# Compute means row
def _mean_col(col):
    vals = [float(v) for v in col if v != 'N/A']
    return f'{sum(vals)/len(vals):.2f}' if vals else 'N/A'

means = {col: _mean_col(summary_df[col]) for col in summary_df.columns if col != 'Term'}
means['Term'] = 'MEAN'
summary_df = pd.concat([summary_df, pd.DataFrame([means])], ignore_index=True)

print('### Summary Delta Table')
display(summary_df.style
    .set_caption('Barrett Construction-Domain Benchmark — Three-Way Model Comparison')
    .set_properties(**{'text-align': 'center'})
    .applymap(lambda v: 'color: green; font-weight: bold' if isinstance(v, str) and v.startswith('+') else '', subset=['Gen Δ (C−A)', 'Judge Δ (B−A)'])
)

---
## Section 6 — Barrett Alignment Commentary

### Interpreting the Generator Delta (C − A)

A positive **Gen Δ** means the generator-tuned model produced simulations that the base judge scored higher. This is the direct empirical signal for Barrett's claim: training on goal-indexed, context-anchored `(context, goal) → simulation` pairs — where only high-adequacy instances (score ≥ 8.0) were used as targets — produces a model that constructs more functionally adequate concept instances in the construction domain.

The mechanism is Barrett-aligned: the training data encodes the population of high-adequacy instances for each term. The model learns, from those examples, what it means to simulate a construction concept in a specific (context, goal) frame — not just to produce fluent text about the domain.

### Interpreting the Judge Delta (B − A)

A non-zero **Judge Δ** means the judge-tuned model assigns different scores than the base model for the same simulation. If the judge-tuned model was calibrated correctly during training, its scores should correlate more tightly with the construction-domain expert ground truth embedded in the training data labels.

A higher Judge Δ on domain-appropriate simulations and a lower (or negative) Judge Δ on generic simulations would be the strongest possible result — the tuned judge has learned to distinguish construction-domain adequacy from generic fluency.

### Why Both Deltas Together Matter

The cross-product design (A/B/C/D) is essential. Without it:
- A generator delta alone could be explained by the tuned generator being more verbose or fluent (not more adequate).
- A judge delta alone could be explained by the tuned judge being biased toward higher scores.

When both are positive and Combination D (tuned generator + tuned judge) is the highest, we have convergent evidence that:
1. The training data produced a generator that constructs better construction-domain simulations.
2. The training data produced a judge that evaluates them more accurately.
3. Both improvements trace to the same Barrett-structured corpus — the data generation strategy itself embodies the theory.

---

*Built with IBM Bob + watsonx.ai · IBM TechXchange Hackathon 2026*